In [ ]:
import numpy as np
import dataset
import dataset_misc1d
from backprop import library
from symbols.const import ConstantSyntaxTree
from symbols.var import VariableSyntaxTree
from symbols.binop import BinaryOperatorSyntaxTree
from gp.utils import replace_subtree
np.seterr(all='ignore')

SAMPLE_SIZE = 100
NOISE = 0 #0.05

In [ ]:
S = dataset_misc1d.MagmanDataset()
S.sample(size=SAMPLE_SIZE, noise=NOISE, mesh=False)
#S.load('../data/magman.csv')
#S.split()
S.get_plotter().plot(width=8, height=6, plot_knowldege=False)

S_train = dataset.NumpyDataset(S)
S_test  = dataset.NumpyDataset(S, test=True)

In [ ]:
lib = library.StaticLibrary(3, S_train, S_train.knowledge)
l = lib.query(S_train.y)
print(l)
l.clear_output()
S.get_plotter().plot(width=8, height=6, model=l)

In [ ]:
backprop_node = ConstantSyntaxTree(2.0)
stree = BinaryOperatorSyntaxTree('/',
        BinaryOperatorSyntaxTree('*',
            ConstantSyntaxTree(0.05),
            VariableSyntaxTree(),
        ),
        backprop_node
    )

stree.set_parent()
y = stree(S_train.X)  # needed for 'pull_output'.
pulled_y = backprop_node.pull_output(S_train.y)

S_backprop = dataset.NumpyDataset(S)
S_backprop.X = S_train.X
S_backprop.y = pulled_y

l = lib.query(pulled_y)
l.clear_output()
print(l)
S_backprop.get_plotter().plot(width=8, height=6, model=l)

In [ ]:
offspring = replace_subtree(stree, backprop_node, l)
offspring.clear_output()
print(offspring)
S.get_plotter().plot(width=8, height=6, plot_knowldege=False, model=offspring, zoomout=1)